## Environment Setup
Install dependent packages and define helper functions.

We assume you have already have pytorch, numpy and tqdm installed. For reproducibility, we depend on the following packages:

[Diffusers](https://huggingface.co/docs/diffusers/en/index) version 0.12.1, you can install it by running following block if
you haven't already.

In [ ]:
!pip install diffusers==0.12.1
!pip install huggingface_hub==0.20.3
!pip uninstall accelerate peft jax jaxlib -y

Here we define some helper functions to display the generated samples in notebook (display_sampl()) and
a "random" noise generator (random_tensor). There are also the reproducibility code, you must run this before
you submit your notebook.

In [ ]:
# !pip uninstall transformers -y

In [ ]:
from diffusers import UNet2DModel
from diffusers import DDPMScheduler
import tqdm
import torch
import PIL.Image
import numpy as np


def display_sample(sample, i):
    """
    Display the processed image sample.
    Args:
        sample (torch.Tensor): The processed image sample.
        i (int): The step number.
    """
    image_processed = sample.cpu().permute(0, 2, 3, 1)
    image_processed = (image_processed + 1.0) * 127.5
    image_processed = image_processed.numpy().astype(np.uint8)

    image_pil = PIL.Image.fromarray(image_processed[0])
    display(f"Image at step {i}")
    display(image_pil)

def randn_tensor(shape, generator=None, device=None, dtype=None, layout=None):
    rand_device = device

    layout = layout or torch.strided
    device = device or torch.device("cpu")

    return torch.randn(
        shape,
        generator=generator,
        device=rand_device,
        dtype=dtype,
        layout=layout,
    ).to(device)

Here is the way to download pre-trained diffusion model from HuggingFace, you can use similar code to download different
diffusion models by exploring their [hub](https://huggingface.co/models).

The reconstruction experiment uses the pretrained `google/ddpm-church-256` model.

In [ ]:
torch.manual_seed(0)

repo_id = "google/ddpm-church-256"
model = UNet2DModel.from_pretrained(repo_id)
model.to("cuda")

noisy_sample = torch.randn(
    1,
    model.config.in_channels,
    model.config.sample_size,
    model.config.sample_size,
)

## Forward diffusion scheduler

q-sample scheduler is used to create ground truth $x_t$ for training the noise network (DDPM) or score network (Score DM).
Here we define a class used to create $x_t$ for $t \in [0, ..., 1000]$.

Here, we define a class called `CustomDDPMScheduler_q` to implement q-sample. This class has three functions:
`sqrt_alpha_prod()`
`sqrt_one_minus_alpha_prod()`
and the main interface function called `add_noise()`.

The forward scheduler implements the DDPM q-sampling equations directly.
You may find Equation 2 and 4 in [DDPM](https://proceedings.neurips.cc/paper/2020/file/4c5bcfec8584af0d967f1ab10179ca4b-Paper.pdf) help.

In [ ]:
class CustomDDPMScheduler_q:
    def __init__(
        self,
        num_train_timesteps,
        beta_start,
        beta_end,
    ):
        # This the basic setting for a 1000 tiemsteps Diffusion Model, and we have
        # defined the betas and alphas and alphas_cumprod for you.
        self.num_inference_steps = 1000
        self.timesteps = torch.from_numpy(
            np.arange(0, num_train_timesteps)[::-1].copy()
        )

        self.betas = torch.linspace(
            beta_start, beta_end, num_train_timesteps
        ).to("cuda")
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = (1 - self.betas).cumprod(dim=0)

    def sqrt_alpha_prod(self, timesteps, original_samples):
        # Extract sqrt of cumulative product of alphas at given timesteps
        # Reshape to (batch, 1, 1, 1) for broadcasting
        sqrt_alpha_prod = self.alphas_cumprod[timesteps] ** 0.5
        return sqrt_alpha_prod.flatten().view(original_samples.shape[0], 1, 1, 1)

    def sqrt_one_minus_alpha_prod(self, timesteps, original_samples):
        # Extract sqrt(1 - cumulative product of alphas) at given timesteps
        # Reshape to (batch, 1, 1, 1) for broadcasting
        sqrt_one_minus_alpha_prod = (1 - self.alphas_cumprod[timesteps]) ** 0.5
        return sqrt_one_minus_alpha_prod.flatten().view(original_samples.shape[0], 1, 1, 1)

    def add_noise(
        self,
        original_samples,
        noise,
        timesteps,
    ):
        # Calculate the square roots of alpha and (1 - alpha) cumulative products
        sqrt_alpha_prod = self.sqrt_alpha_prod(timesteps, original_samples)
        sqrt_one_minus_alpha_prod = self.sqrt_one_minus_alpha_prod(timesteps, original_samples)

        # Implementation of Equation 4: x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * epsilon
        noisy_samples = sqrt_alpha_prod * original_samples + sqrt_one_minus_alpha_prod * noise
        return noisy_samples

In [ ]:
# # p sample
# scheduler = CustomDDPMScheduler_q(1000, 0.0001, 0.02)

# # NOTE: Here you must use the npy file which you can find with this notebook!
# sample = np.load("tensor.npy")
# noisy_sample = torch.tensor(sample)

# # Here is how you use the custom DDPMScheduler to destory image into the Noise:
# noise = torch.randn(
#     noisy_sample.shape,
#     dtype=torch.float32,
#     device="cuda" # Add this
# )
# noisy_sample = scheduler.add_noise(
#     noisy_sample, noise, timesteps=torch.tensor(999, dtype=torch.int32)
# )
# np.save("noisy_sample.npy", noisy_sample.numpy())
# noisy_sample = noisy_sample.to("cuda")
# sample = noisy_sample.clone()

In [ ]:
# I have changed this...

# p sample
scheduler = CustomDDPMScheduler_q(1000, 0.0001, 0.02)

# NOTE: Here you must use the npy file which you can find with this notebook!
sample = np.load("tensor.npy")

# 1. Create the tensor and move it to CUDA immediately
noisy_sample = torch.tensor(sample).to("cuda") 

# 2. Ensure noise is created on the same device
noise = torch.randn(
    noisy_sample.shape,
    dtype=torch.float32,
    device="cuda" # Add this
)

# 3. Ensure timesteps is also on the same device
timesteps = torch.tensor(999, dtype=torch.int32).to("cuda")

noisy_sample = scheduler.add_noise(
    noisy_sample, noise, timesteps=timesteps
)

# 4. To save with numpy, you must move it back to CPU
np.save("noisy_sample.npy", noisy_sample.cpu().numpy()) 

# The rest of your code
sample = noisy_sample.clone()

## Reverse diffusion scheduler

p-sample scheduler is the way how we use the pre-trained UNet (noise network and score network)
to reconstruct image from the noise.

The reverse scheduler implements the DDPM p-sampling equations directly:
1. Have a deeper look at the original paper of [DDPM]((https://proceedings.neurips.cc/paper/2020/file/4c5bcfec8584af0d967f1ab10179ca4b-Paper.pdf).
2. You should follow the name of the variable, that the HINT to you.
3. 

In [ ]:
class CustomDDPMScheduler_p:
    def __init__(
        self,
        num_train_timesteps,
        beta_start,
        beta_end,
    ):
        # This is the same as the previous CustomDDPMScheduler_q
        self.num_inference_steps = 1000
        self.timesteps = torch.from_numpy(
            np.arange(0, num_train_timesteps)[::-1].copy()
        )

        self.betas = torch.linspace(
            beta_start, beta_end, num_train_timesteps
        ).to("cuda")
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = (1 - self.betas).cumprod(dim=0)

    def get_variance(self, t):
        alpha_prod_t = self.alphas_cumprod[t]
        alpha_prod_t_prev = self.alphas_cumprod[t - 1] if t > 0 else torch.tensor(1.0).to("cuda")
        
        # Calculation of sigma_t^2
        variance = (1 - alpha_prod_t_prev) / (1 - alpha_prod_t) * self.betas[t]
        return torch.clamp(variance, min=1e-20)

    def compute_alphas_betas(self, t):
        alpha_prod_t = self.alphas_cumprod[t]
        alpha_prod_t_prev = self.alphas_cumprod[t - 1] if t > 0 else torch.tensor(1.0).to("cuda")
        beta_prod_t = 1 - alpha_prod_t
        beta_prod_t_prev = 1 - alpha_prod_t_prev
        return alpha_prod_t, alpha_prod_t_prev, beta_prod_t, beta_prod_t_prev

    def predict_original_sample(self, model_output, sample, beta_prod_t, alpha_prod_t):
        pred_original_sample = (sample - beta_prod_t ** 0.5 * model_output) / (alpha_prod_t ** 0.5)
        return torch.clamp(pred_original_sample, -1, 1)

    def compute_coeffs(self, t, beta_prod_t, alpha_prod_t, alpha_prod_t_prev):
        # Coefficients for x_0 and x_t
        pred_original_sample_coeff = (alpha_prod_t_prev ** 0.5 * self.betas[t]) / beta_prod_t
        current_sample_coeff = (self.alphas[t] ** 0.5 * (1 - alpha_prod_t_prev)) / beta_prod_t
        return pred_original_sample_coeff, current_sample_coeff

    def predict_previous_sample(self, pred_original_sample_coeff, current_sample_coeff, pred_original_sample, sample):
        # The "?" in the template refers to the addition (+) and multiplication (*) operators
        mu_t = pred_original_sample_coeff * pred_original_sample + current_sample_coeff * sample
        return mu_t

    def step(self, model_output, timestep, sample):
        t = timestep
        alpha_prod_t, alpha_prod_t_prev, beta_prod_t, beta_prod_t_prev = self.compute_alphas_betas(t)
        pred_original_sample = self.predict_original_sample(model_output, sample, beta_prod_t, alpha_prod_t)
        pred_original_sample_coeff, current_sample_coeff = self.compute_coeffs(t, beta_prod_t, alpha_prod_t, alpha_prod_t_prev)
        pred_prev_sample = self.predict_previous_sample(pred_original_sample_coeff, current_sample_coeff, pred_original_sample, sample)

        # 6. Add noise (as in the original implementation)
        variance = 0
        if t > 0:
            device = model_output.device
            variance_noise = randn_tensor(
                model_output.shape,
                generator=None,
                device=device,
                dtype=model_output.dtype,
            )
            variance = (self.get_variance(t) ** 0.5) * variance_noise

        pred_prev_sample = pred_prev_sample + variance

        return pred_prev_sample

### Here, we show you have to use Diffusers' API to reconstruct a image from noise.

In [ ]:
torch.manual_seed(0)
sample = torch.tensor(np.load("noisy_sample.npy")).to("cuda")
diffusers_scheduler = DDPMScheduler.from_config(repo_id)
for i, t in enumerate(tqdm.tqdm(scheduler.timesteps)):
    # 1. predict noise residual
    with torch.no_grad():
        residual = model(sample, t).sample

    # 2. compute less noisy image and set x_t -> x_t-1
    sample = diffusers_scheduler.step(residual, t, sample).prev_sample

    # 3. optionally look at image
    if (i + 1) % 100 == 0:
        display_sample(sample, i + 1)

diffusers_sample = sample.cpu().clone()

### Here is how you use your custom Scheduler to reconstruct the image.

You should exptected your reconstructed image is the same as the Diffusers',
and if they are different, you should restart your session and rerun.

In [ ]:
# q sample
torch.manual_seed(0)
scheduler = CustomDDPMScheduler_p(1000, 0.0001, 0.02)
sample = torch.tensor(np.load("noisy_sample.npy")).to("cuda")
for i, t in enumerate(tqdm.tqdm(scheduler.timesteps)):
    # 1. predict noise residual
    with torch.no_grad():
        residual = model(sample, t).sample

    # 2. compute less noisy image and set x_t -> x_t-1
    sample = scheduler.step(residual, t, sample)

    # 3. optionally look at image
    if (i + 1) % 100 == 0:
        display_sample(sample, i + 1)

our_sample = sample.cpu().clone()